# paper-translator: Colab サーバー

GPU ランタイムで実行する。VS Code の Colab 拡張機能でカーネルに Colab（L4 / A100）を選んでも、ブラウザの Colab でも動く。
カーネルは Colab のマシン上で動くため、コードは GitHub から取得する（手元の変更は push → 取得セルを再実行 → カーネル再起動）。

画面と API を起動して cloudflared で公開し、最後に表示される「アプリを開く」のリンクを開く（API URL は自動で入る）。

**GPU の選び方**: モデルカードは BF16 対応 GPU（L4 / A100 など）を前提にしている。T4 は BF16 非対応で、fp16 で動くかは未確認。

| モデル | BF16 の重みサイズ |
|---|---|
| `pfnet/plamo-3-translate-2b`（既定） | 約 5.8 GB |
| `pfnet/plamo-3-translate-8b` | 約 17.1 GB |


In [ ]:
!nvidia-smi

In [ ]:
import os

REPO = "/content/paper-translator"
if os.path.isdir(REPO):
    !git -C {REPO} pull -q
else:
    !git clone -q https://github.com/Kame09-M/paper-translator.git {REPO}
%cd {REPO}/backend
!git log --oneline -1
!pip install -q -U -r requirements-gpu.txt

# 注: pull で取り込んだコードの変更は、カーネルを再起動してから反映される
# （Colab の IPython の autoreload は Python 3.13 で動かない: No module named 'imp'）

## 画面をビルド

画面（frontend）をビルドして、API サーバーから一緒に配信できるようにする（1〜2分）。
Node.js は取得セルの pip（`nodejs-wheel`）で入る。


In [ ]:
!cd ../frontend && npm ci --no-audit --no-fund --loglevel=error && npm run build

## 設定

Gemini の API キーは、アプリの画面上部の「Gemini API キー」欄に入れる（ブラウザに保存され、要約・用語説明のときだけサーバーに送られる）。
ノートブックには書かない。画面で空のままなら、用語説明と要約はダミーで動く。


In [ ]:
import os

os.environ["MOCK"] = "0"  # 翻訳を本物にする
os.environ["PLAMO_MODEL"] = "pfnet/plamo-3-translate-2b"

## 翻訳だけ先に試す

モデルの読み込み（初回はダウンロードあり）と、履歴（直前の段落）付きの翻訳を確かめる。


In [ ]:
from app.services import translate

paras = [
    "The Transformer relies entirely on attention to draw global dependencies.",
    "Self-attention relates different positions of a single sequence.",
]
for src, tgt in zip(paras, translate.translate_paragraphs(paras, "ja")):
    print(src, "\n→", tgt, "\n")

## 実物の論文で確かめる（時間・要約・文脈の効果）

公開論文「Attention Is All You Need」で、要約（Gemini）と、先頭 N 段落の翻訳時間を測る。
直前の段落（文脈）あり／なしの訳も並べるので、目で見比べる（簡易評価）。


In [ ]:
import time
import urllib.request

from app.services import pdf, summarize

urllib.request.urlretrieve("https://arxiv.org/pdf/1706.03762", "/content/attention.pdf")
paragraphs = pdf.extract_paragraphs(open("/content/attention.pdf", "rb").read())
print("段落数:", len(paragraphs))

t = time.time()
print(summarize.summarize_paper(paragraphs, "ja"))
print(f"要約: {time.time() - t:.1f} 秒")

In [ ]:
N = 5
sample = paragraphs[4 : 4 + N]  # 冒頭の注記を飛ばす

t = time.time()
with_context = translate.translate_paragraphs(sample, "ja")
sec = (time.time() - t) / N
print(f"1段落あたり {sec:.1f} 秒 → 全 {len(paragraphs)} 段落なら約 {sec * len(paragraphs) / 60:.1f} 分")

without_context = [translate.translate_paragraph(p, "ja") for p in sample]

for src, a, b in zip(sample, with_context, without_context):
    print("原文:", src[:200], "\n文脈あり:", a, "\n文脈なし:", b, "\n")

## API サーバー起動

起動後、Colab の中から1回翻訳を呼んでモデルを読み込んでおく（初回はダウンロード込みで数分）。
読み込まずに公開すると、画面からの最初のリクエストがモデル読み込みを待って cloudflared の時間制限（約100秒）で切れるため。


In [ ]:
import json
import subprocess
import time
import urllib.request

server = subprocess.Popen(
    ["uvicorn", "app.main:app", "--host", "0.0.0.0", "--port", "8000"],
    stdout=open("server.log", "w"),
    stderr=subprocess.STDOUT,
)

for _ in range(60):  # 起動待ち
    try:
        print(urllib.request.urlopen("http://localhost:8000/api/health").read().decode())
        break
    except Exception:
        time.sleep(1)
else:
    print(open("server.log").read())
    raise RuntimeError("サーバーが起動しない（上のログを確認）")

t = time.time()
req = urllib.request.Request(
    "http://localhost:8000/api/translate-paragraph",
    json.dumps({"text": "Hello.", "target_lang": "ja"}).encode(),
    {"Content-Type": "application/json"},
)
print(json.load(urllib.request.urlopen(req, timeout=1800)), f"モデル読み込み込み {time.time() - t:.0f} 秒")

## cloudflared で公開

最後に出る「アプリを開く」のリンクを開く。画面も API もこの URL から配信されるので、API URL 欄は自動で入る。
cloudflared は裏で動き続ける（カーネルを止めると止まる）。リンクを開いてすぐはつながらないことがあるので、数秒待って再読み込みする。


In [ ]:
import re
import subprocess
import time

from IPython.display import Markdown, display

!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared && chmod +x cloudflared

# 裏で動かし、ログから公開 URL を拾う（ログはファイルに書くので、出力が詰まって止まることがない）
tunnel = subprocess.Popen(
    ["./cloudflared", "tunnel", "--url", "http://localhost:8000"],
    stdout=open("cloudflared.log", "w"),
    stderr=subprocess.STDOUT,
)
url = None
for _ in range(60):
    found = re.search(r"https://(?!api\.)[a-z0-9-]+\.trycloudflare\.com", open("cloudflared.log").read())
    if found:
        url = found.group(0)
        break
    time.sleep(1)
if url is None:
    print(open("cloudflared.log").read())
    raise RuntimeError("cloudflared の URL が出ない（上のログを確認）")
display(Markdown(f"**アプリを開く**: [{url}]({url})"))